# Patient Data Storage Design

This notebook stores ICU data in two scalable tables:
- **patients_static**: one row per patient (Age, Gender, Height, ICUType, Weight).
- **patient_events**: long-format time-series table for dynamic variables (HR, Temp, labs, urine, etc.).

This pattern works well for many patients because dynamic measurements are kept in one append-friendly table.

In [2]:
from pathlib import Path
import pandas as pd

STATIC_PARAMETERS = {"RecordID", "Age", "Gender", "Height", "Weight"}

def hhmm_to_minutes(value: str) -> int:
    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

def parse_patient_file(file_path: Path):
    df = pd.read_csv(file_path)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    record_rows = df.loc[df["Parameter"] == "RecordID", "Value"].dropna()
    patient_id = int(record_rows.iloc[0]) if not record_rows.empty else int(file_path.stem)

    static_df = df[df["Parameter"].isin(STATIC_PARAMETERS)].copy()
    static_df = static_df.drop_duplicates(subset=["Parameter"], keep="last")
    static_values = static_df.set_index("Parameter")["Value"].to_dict()

    static_record = {
        "RecordID": patient_id,
        "Age": static_values.get("Age"),
        "Gender": static_values.get("Gender"),
        "Height": static_values.get("Height"),
        "ICUType": static_values.get("ICUType"),
        "Weight": static_values.get("Weight"),
    }

    dynamic_df = df[~df["Parameter"].isin(STATIC_PARAMETERS)].copy()
    dynamic_df["Minutes"] = dynamic_df["Time"].map(hhmm_to_minutes)
    dynamic_df["RecordID"] = patient_id
    dynamic_df = dynamic_df[["RecordID", "Time", "Minutes", "Parameter", "Value"]]

    return static_record, dynamic_df

def load_cohort(folder: str):
    folder_path = Path(folder)
    static_records = []
    dynamic_tables = []

    for file_path in sorted(folder_path.glob("*.txt")):
        static_record, dynamic_df = parse_patient_file(file_path)
        static_records.append(static_record)
        dynamic_tables.append(dynamic_df)

    patients_static = pd.DataFrame(static_records).drop_duplicates(subset=["RecordID"])
    patients_static = patients_static.set_index("RecordID").sort_index()

    if dynamic_tables:
        patient_events = pd.concat(dynamic_tables, ignore_index=True)
        patient_events = patient_events.sort_values(["RecordID", "Minutes", "Parameter"]).reset_index(drop=True)
    else:
        patient_events = pd.DataFrame(columns=["RecordID", "Time", "Minutes", "Parameter", "Value"])

    return patients_static, patient_events

In [3]:
train_data_location = "set-a"
patients_static, patient_events = load_cohort(train_data_location)

print(f"Number of patients: {patients_static.shape[0]}")
print(f"Number of dynamic observations: {patient_events.shape[0]}")

display(patients_static.head())
display(patient_events.head(15))

Number of patients: 4000
Number of dynamic observations: 1608815


,Age,Gender,Height,ICUType,Weight
RecordID,,,,,
132539,54.0,0.0,-1.0,4.0,-1.0
132540,76.0,1.0,175.3,2.0,81.6
132541,44.0,0.0,-1.0,3.0,56.7
132543,68.0,1.0,180.3,3.0,84.6
132545,88.0,0.0,-1.0,3.0,-1.0


,RecordID,Time,Minutes,Parameter,Value
0,132539,00:07,7,GCS,15.00
1,132539,00:07,7,HR,73.00
2,132539,00:07,7,NIDiasABP,65.00
3,132539,00:07,7,NIMAP,92.33
4,132539,00:07,7,NISysABP,147.00
5,132539,00:07,7,RespRate,19.00
6,132539,00:07,7,Temp,35.10
7,132539,00:07,7,Urine,900.00
8,132539,00:37,37,HR,77.00
9,132539,00:37,37,NIDiasABP,58.00


In [6]:
# Optional dictionary view for convenient per-patient access
patient_store = {
    patient_id: {
        "static": patients_static.loc[patient_id].to_dict(),
        "dynamic": group.drop(columns=["RecordID"]).reset_index(drop=True),
    }
    for patient_id, group in patient_events.groupby("RecordID")
}

sample_patient_id = next(iter(patient_store))
print(f"Example patient id: {sample_patient_id}")
print("Static fields:", patient_store[sample_patient_id]["static"])
display(patient_store[sample_patient_id]["dynamic"].head())

Example patient id: 132539
Static fields: {'Age': 54.0, 'Gender': 0.0, 'Height': -1.0, 'ICUType': 4.0, 'Weight': -1.0}


,Time,Minutes,Parameter,Value
0,00:07,7,GCS,15.00
1,00:07,7,HR,73.00
2,00:07,7,NIDiasABP,65.00
3,00:07,7,NIMAP,92.33
4,00:07,7,NISysABP,147.00


In [7]:
# Missing-value audit for static + dynamic tables
static_nan_counts = patients_static.isna().sum().sort_values(ascending=False)
event_nan_counts = patient_events.isna().sum().sort_values(ascending=False)

# In this dataset, -1 is often used as a sentinel for unknown static values.
static_minus1_counts = (patients_static == -1).sum().sort_values(ascending=False)

print("NaN counts in patients_static:")
display(static_nan_counts.to_frame("nan_count"))

print("-1 sentinel counts in patients_static:")
display(static_minus1_counts.to_frame("minus1_count"))

print("NaN counts in patient_events:")
display(event_nan_counts.to_frame("nan_count"))

print(f"Rows with NaN Value in patient_events: {patient_events['Value'].isna().sum()}")
print(f"Rows with NaN Time in patient_events: {patient_events['Time'].isna().sum()}")
print(f"Rows with NaN Parameter in patient_events: {patient_events['Parameter'].isna().sum()}")

NaN counts in patients_static:


,nan_count
Age,0
Gender,0
Height,0
ICUType,0
Weight,0


-1 sentinel counts in patients_static:


,minus1_count
Height,1894
Weight,296
Gender,3
Age,0
ICUType,0


NaN counts in patient_events:


,nan_count
RecordID,0
Time,0
Minutes,0
Parameter,0
Value,0


Rows with NaN Value in patient_events: 0
Rows with NaN Time in patient_events: 0
Rows with NaN Parameter in patient_events: 0


In [8]:
from pathlib import Path

def save_patient_cache(patients_static, patient_events, out_dir="cache"):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    static_path = out_path / "patients_static.parquet"
    events_path = out_path / "patient_events.parquet"

    patients_static.reset_index().to_parquet(static_path, index=False)
    patient_events.to_parquet(events_path, index=False)

    print(f"Saved: {static_path}")
    print(f"Saved: {events_path}")

def load_patient_cache(out_dir="cache"):
    out_path = Path(out_dir)
    static_path = out_path / "patients_static.parquet"
    events_path = out_path / "patient_events.parquet"

    if not static_path.exists() or not events_path.exists():
        raise FileNotFoundError(
            f"Cache files not found in {out_path}. Run save_patient_cache(...) first."
        )

    patients_static_cached = pd.read_parquet(static_path).set_index("RecordID").sort_index()
    patient_events_cached = pd.read_parquet(events_path)
    patient_events_cached = patient_events_cached.sort_values(
        ["RecordID", "Minutes", "Parameter"]
    ).reset_index(drop=True)

    return patients_static_cached, patient_events_cached

# First-time save (run once after loading raw files)
save_patient_cache(patients_static, patient_events, out_dir="cache")

# Fast reload path (run in future sessions instead of parsing all txt files)
patients_static_cached, patient_events_cached = load_patient_cache(out_dir="cache")
print(
    f"Cached load check -> patients: {patients_static_cached.shape[0]}, "
    f"events: {patient_events_cached.shape[0]}"
 )

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.